# Silver — Food Inspections (dev) — v3 (Delta)

Typed, validated, deduplicated layer over the **bronze Delta table**:

* **Struct-typed schema** enforced on every row with ANSI-safe `try_cast` (failed conversions become NULL). Extra bronze columns (`:id`, `:created_at`, `:version`, `_ingest_date`) are dropped intentionally — `:updated_at` is kept (TIMESTAMP) as the SCD ordering key, and ingestion lineage is tracked via `_source` / `_ingested_at`.
* **Required fields**: `inspection_id, dba_name, license_, risk, inspection_date, results` — rows missing any of them (including failed casts) go to the quarantine table `foodguard_ai.silver.food_inspections_quarantine`
* **Deduplication**: unique on `(inspection_id, inspection_date, results)`
* **`record_id`**: autogenerated `sha2` hash of `(inspection_id, inspection_date, results)` — unique across the entire silver layer
* **`is_active`**: SCD flag by the source update timestamp — the newest version (`:updated_at`, ties broken by `_ingested_at`) of each inspection `(inspection_id, inspection_date)` is active; superseded versions are kept with `is_active = false`

Reads ALL append-only bronze history, fully recomputes the silver snapshot, and **MERGE-upserts it into the Delta table** `foodguard_ai.silver.food_inspections` keyed on `record_id` (matched → update, new → insert) — deterministic and idempotent on re-run, with only changed rows rewritten. Future gold layer: `foodguard_ai.gold.food_inspections`.

In [0]:
# Layer tables (Delta Lake) and quarantine.
BRONZE_TABLE = "foodguard_ai.bronze.food_inspections"
SILVER_TABLE = "foodguard_ai.silver.food_inspections"
QUARANTINE_TABLE = "foodguard_ai.silver.food_inspections_quarantine"

print(f"Bronze     : {BRONZE_TABLE}")
print(f"Silver     : {SILVER_TABLE}")
print(f"Quarantine : {QUARANTINE_TABLE}")

In [0]:
# Peek at the bronze data: structure + sample values (all STRING, plus audit columns).
bronze_df = spark.table(BRONZE_TABLE)

bronze_df.printSchema()
print(f"Rows: {bronze_df.count()}")
display(bronze_df.limit(10))

In [0]:
# Silver schema as a StructType, informed by the peek above: numeric IDs and zip
# are longs, inspection_date is a timestamp; free-text and partially malformed
# columns (latitude / longitude / location, names) stay STRING.
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType,
)

SILVER_SCHEMA = StructType([
    StructField("inspection_id",   LongType(),      True),
    StructField("dba_name",        StringType(),    True),
    StructField("aka_name",        StringType(),    True),
    StructField("license_",        LongType(),      True),
    StructField("facility_type",   StringType(),    True),
    StructField("risk",            StringType(),    True),
    StructField("address",         StringType(),    True),
    StructField("city",            StringType(),    True),
    StructField("state",           StringType(),    True),
    StructField("zip",             LongType(),      True),
    StructField("inspection_date", TimestampType(), True),
    StructField("inspection_type", StringType(),    True),
    StructField("results",         StringType(),    True),
    StructField("violations",      StringType(),    True),
    StructField("latitude",        StringType(),    True),
    StructField("longitude",       StringType(),    True),
    StructField("location",        StringType(),    True),
    StructField("_source",         StringType(),    True),
    StructField("_ingested_at",    TimestampType(), True),  # ISO string -> timestamp, ingestion lineage
    StructField(":updated_at",      TimestampType(), True),  # source-system update time — SCD ordering key
])

# Enforce the struct schema on ALL rows. try_cast is ANSI-safe: a failed
# conversion becomes NULL (a NULL in a required field quarantines the row).
# Columns not in the schema (system fields :id / :created_at / :version, _ingest_date) are
# dropped here; :updated_at is kept, typed TIMESTAMP, as the SCD ordering key.
silver_df = bronze_df.select([
    F.expr(f"try_cast(`{f.name}` as {f.dataType.simpleString()})").alias(f.name)
    for f in SILVER_SCHEMA.fields
])

silver_df.printSchema()

In [0]:
# Required fields: a row missing ANY of them (null in bronze, or null because the
# cast failed) is quarantined to the Unity Catalog table. Raw pre-cast values
# always remain in bronze for forensics.
REQUIRED = ["inspection_id", "dba_name", "license_", "risk", "inspection_date", "results"]
_all_not_null = " AND ".join(f"`{c}` IS NOT NULL" for c in REQUIRED)
_any_null = " OR ".join(f"`{c}` IS NULL" for c in REQUIRED)

valid_df = silver_df.where(_all_not_null)

quarantine_records = (
    silver_df.where(_any_null)
    .withColumn(
        "reason",
        F.concat_ws(", ", *[F.when(F.col(c).isNull(), F.lit(c)) for c in REQUIRED]),
    )
    .withColumn("quarantined_at", F.current_timestamp())
    .dropDuplicates(silver_df.columns)
)

# Create the quarantine table in the catalog (idempotent).
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {QUARANTINE_TABLE} (
        {", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in SILVER_SCHEMA.fields)},
        `reason` STRING,
        `quarantined_at` TIMESTAMP
    )
""")

# The quarantine table may predate the `:updated_at` column; evolve it (idempotent).
if ":updated_at" not in {c.name for c in spark.table(QUARANTINE_TABLE).schema.fields}:
    spark.sql(f"ALTER TABLE {QUARANTINE_TABLE} ADD COLUMN `:updated_at` TIMESTAMP")

quarantined_count = quarantine_records.count()
if quarantined_count > 0:
    quarantine_records.write.mode("append").saveAsTable(QUARANTINE_TABLE)
    print(f"Quarantined to {QUARANTINE_TABLE}: {quarantined_count} rows "
          f"(re-running re-appends the same rows — quarantine is an audit trail)")
else:
    print("No rows missing required fields — nothing quarantined.")

print(f"Valid rows: {valid_df.count()}")

In [0]:
# record_id: deterministic sha2 hash of the dedup key (inspection_id,
# inspection_date, results). The dedup rule — unique (inspection_id,
# inspection_date, results) — is exactly uniqueness on record_id.
# Dedup spans ALL bronze history (append-only), so repeated ingests collapse.
batch_df = valid_df.withColumn(
    "record_id",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("inspection_id").cast("string"),
            F.col("inspection_date").cast("string"),
            F.col("results"),
        ),
        256,
    ),
).dropDuplicates(["record_id"])

print(f"Rows after dedup: {batch_df.count()} (from {valid_df.count()} valid)")

In [0]:
# SCD logic on the SOURCE update timestamp (:updated_at), computed over ALL bronze history:
#   * the newest source update of each inspection (inspection_id, inspection_date) is active
#   * ties on :updated_at are broken by _ingested_at (latest ingest wins), so the
#     snapshot stays deterministic across re-runs
from pyspark.sql.window import Window

scd_window = Window.partitionBy("inspection_id", "inspection_date").orderBy(
    F.col(":updated_at").desc(), F.col("_ingested_at").desc()
)
final_df = (
    batch_df.withColumn("_rn", F.row_number().over(scd_window))
    .withColumn("is_active", F.col("_rn") == 1)
    .drop("_rn")
)

# Upsert the recomputed snapshot via MERGE on record_id:
#   * matched record_id -> update every column (e.g. is_active flips when a newer version arrives)
#   * new record_id     -> insert
# Bronze is append-only, so every record_id ever seen is always in the snapshot —
# nothing in the target is ever orphaned, and only changed rows are rewritten.
final_df.createOrReplaceTempView("silver_snapshot")

if not spark.catalog.tableExists(SILVER_TABLE):   # MERGE needs an existing target
    final_df.write.mode("overwrite").saveAsTable(SILVER_TABLE)   # first run: create from the snapshot

spark.sql(f"""
    MERGE INTO {SILVER_TABLE} AS t
    USING silver_snapshot AS s
        ON t.record_id = s.record_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

table_df = spark.table(SILVER_TABLE)
print(f"Merged into {SILVER_TABLE}: {table_df.count()} rows "
      f"({table_df.where(F.col('is_active')).count()} active)")

In [0]:
# Final sanity: row counts, record_id uniqueness in the silver table, required-field
# completeness, schema, SCD summary, and quarantine total.
check_df = spark.table(SILVER_TABLE)

out_count, final_count = check_df.count(), final_df.count()
unique_layer = check_df.select("record_id").distinct().count() == out_count
active_count = check_df.where(F.col("is_active")).count()
required_violations = check_df.where(_any_null).count()
schema_ok = check_df.dtypes == final_df.dtypes  # types persisted (nullability differs after Delta round-trip)
quarantine_total = spark.table(QUARANTINE_TABLE).count()

print(f"Rows             — final: {final_count} | table: {out_count} | match: {final_count == out_count}")
print(f"record_id unique — entire silver table: {unique_layer}")
print(f"SCD              — active: {active_count} | inactive: {out_count - active_count}")
print(f"Required fields  — violations in silver: {required_violations}")
print(f"Schema persisted : {schema_ok}")
print(f"Quarantine table rows (all runs): {quarantine_total}")
assert final_count == out_count, "Row count mismatch!"
assert unique_layer, "record_id not unique across the silver layer!"
assert required_violations == 0, "Required fields missing in silver!"
assert schema_ok, "Schema mismatch — enforced types were not persisted!"
display(check_df.limit(5))